In [1]:
from main import train_epoch
from data import get_data

special_symbols = {
            "pad": {"cont": [0.,1.],"CEL":-150},
            "bos": {"cont": [1.,1.], "CEL":-100},
            "eos": {"cont": [1.,0.],"CEL":-50},
            "sample": [0.,0.]
    }
batch_size = 5
dir_path = "/Users/paulwahlen/Desktop/Internship/ML/Code/TransfoV1/data"
vocab_charges, vocab_pdgs, E_label_RMSNormalizer, data_ld = get_data(dir_path, batch_size, special_symbols, "training")

ValueError: too many values to unpack (expected 2)

In [3]:
import torch
feats5,labels5 =  next(iter(data_ld))
feats5_reduced = feats5[:,::100]
print(feats5_reduced.size(dim = 1))
torch.set_printoptions(profile= "full")
print(feats5_reduced[0])

NameError: name 'data_ld' is not defined

In [3]:
def train_epoch(model, optim, train_dl, special_symbols,vocab_charges, vocab_pdgs, 
          hyperweights_lossfn, loss_fn_charges, loss_fn_pdg,loss_fn_cont):
    model.train() #setting model into train mode
    loss_epoch = 0.0
    for src,tgt in train_dl:
        src.to(DEVICE)
        tgt.to(DEVICE)

        src_padding_mask, tgt_padding_mask = create_mask(src,tgt,special_symbols["pad"]["cont"])
        tgt_in_padding_mask = tgt_padding_mask[:,:-1,:]
        tgt_out_padding_mask = tgt_padding_mask[:,1:,:]

        tgt_in = tgt[:,:-1] #sets the dimensions of transformer output -> must have the same as tgt_out
        logits_charges, logits_pdg, logits_cont = model(src,tgt_in, 
                                                                    src_padding_mask,
                                                                    tgt_in_padding_mask,
                                                                    src_padding_mask)
        optim.zero_grad()

        tgt_out = tgt[:,1:,:] #logits are compared with tokens shifted
        tgt_out_charges = tgt_out[...,0]
        tgt_out_pdg = tgt_out[...,1]
        tgt_out_cont = tgt_out[...,2:-2] #only (E, theta, phi)
        #Using spherical coordinates to get 3D direction vectors
        tgt_out_sin_theta = torch.sin(tgt_out_cont[...,0])
        tgt_out_cont[...,0] = torch.cos(tgt_out_cont[...,1]) * tgt_out_sin_theta
        tgt_out_cont[...,1] = torch.sin(tgt_out_cont[...,1]) * tgt_out_sin_theta
        tgt_out_cont = torch.concat([tgt_out_cont, torch.cos(tgt_out_cont[...,0]).unsqueeze_(-1)], dim = -1)
        #setting values of special tokens to 0 so that it doesn't contribute to loss
        tgt_out_cont[tgt_out_padding_mask] = torch.zeros(tgt_out_cont.size[-1])
        logits_cont[tgt_out_padding_mask] = torch.zeros(tgt_out_cont.size[-1])
        #eos and bos positions:
        eos_bos_mask = ((tgt_out_charges == vocab_charges.get_index(special_symbols["eos"]["CEL"])) 
                        or (tgt_out_charges == vocab_charges.get_index(special_symbols["bos"]["CEL"]))) 
        logits_cont[eos_bos_mask] = 0.0 #not sure if broadcasting works here
        
        #Computing the losses
        loss_charges = loss_fn_charges(logits_charges.transpose(dim0 = -2, dim1 = -1), tgt_out_charges)
        loss_pdg = loss_fn_pdg(logits_pdg.transpose(dim0 = -2, dim1 = -1), tgt_out_pdg)
        loss_cont_vec = loss_fn_cont(logits_cont, tgt_out_cont, reduction = 'none')
        npad = torch.count_nonzero(tgt_out_padding_mask, dim = -1)
        n_nopad = tgt_out_padding_mask.shape[-1] - npad
        loss_cont = torch.mean(loss_cont_vec * n_nopad)

        loss_vec = torch.tensor([loss_charges,loss_pdg, loss_cont])
        loss = torch.dot(torch.tensor(hyperweights_lossfn),loss_vec)

        loss.backward()
        optim.step()

        loss_epoch += loss.item()

    return loss_epoch / len(list(train_dl))

In [17]:
from data import create_mask
src_padding, tgt_padding = create_mask(feats5_reduced, labels5, special_symbols["pad"]["cont"])
print(feats5_reduced[~src_padding].shape)
print(src_padding.shape)

torch.Size([74, 6])
torch.Size([5, 23])


In [14]:
#print(vocab_charges.vocab)
#print(vocab_charges.vocab["-50"])
#print(vocab_charges.vocab.keys())
keys = list(vocab_charges.vocab.keys())
eos = keys[0]
print(eos)
print(vocab_charges.get_index(eos))
print(vocab_charges.get_index(special_symbols["eos"]["CEL"]))
print(vocab_charges.get_index(special_symbols["bos"]["CEL"]))
eos_bos_mask = ((labels5[...,0] == vocab_charges.get_index(special_symbols["eos"]["CEL"])) 
                        + (labels5[...,0] == vocab_charges.get_index(special_symbols["bos"]["CEL"]))) 
print(eos_bos_mask[0])
print(labels5[0])

-150
tensor(0)
tensor(2)
tensor(1)
tensor([ True, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False,  True, False, False, False, False, False, False, False, False,
        False, False])
tensor([[ 1.0000,  1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000],
        [ 3.0000,  4.0000, -0.8577,  0.9286, -0.0454, -0.3683,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4446, -0.5542, -0.7102,  0.4342,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4511, -0.5834, -0.6703,  0.4587,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4468, -0.6857, -0.3811,  0.6201,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4483, -0.0462,  0.9894,  0.1377,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4532, -0.0132,  0.9981,  0.0595,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4469,  0.1253,  0.9801,  0

In [13]:
bool1 = torch.tensor([True, True, False, False])
bool2 = torch.tensor([False, True, False, True])
print(bool1+bool2)

tensor([ True,  True, False,  True])


In [16]:
labels5[eos_bos_mask] = 0.
print(labels5[0])

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 3.0000,  4.0000, -0.8577,  0.9286, -0.0454, -0.3683,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4446, -0.5542, -0.7102,  0.4342,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4511, -0.5834, -0.6703,  0.4587,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4468, -0.6857, -0.3811,  0.6201,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4483, -0.0462,  0.9894,  0.1377,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4532, -0.0132,  0.9981,  0.0595,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4469,  0.1253,  0.9801,  0.1540,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4538,  0.3886,  0.6474,  0.6557,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4539,  0.5872,  0.7305,  0.3486,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4470,  0.8047,  0.2092,  0.5556,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4539, -0.7112, -0.1417,  0.6885,  0.0000,  0.0000],
        [ 3.0000

In [19]:
special_mask = tgt_padding + eos_bos_mask
print(special_mask[0])
labels5[special_mask] = 0.
print(labels5[0])

tensor([ True, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True])
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 3.0000,  4.0000, -0.8577,  0.9286, -0.0454, -0.3683,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4446, -0.5542, -0.7102,  0.4342,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4511, -0.5834, -0.6703,  0.4587,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4468, -0.6857, -0.3811,  0.6201,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4483, -0.0462,  0.9894,  0.1377,  0.0000,  0.0000],
        [ 4.0000,  5.0000, -0.4532, -0.0132,  0.9981,  0.0595,  0.0000,  0.0000],
        [ 3.0000,  7.0000,  1.4469,  0.1253,  0.9801,  0.1540,  0.0000,  0.0000],
        [

In [26]:
nspe_tokens = torch.count_nonzero(special_mask, dim = -1)
n_nospe = special_mask.shape[-1] - nspe_tokens
print(n_nospe)
print(torch.sum(n_nospe))
print(labels5[~special_mask].shape)

tensor([30, 23, 29, 18, 19])
tensor(119)
torch.Size([119, 8])


In [30]:
logits_cont = torch.rand((5,30,3)).fill_(1)
logits_cont_3d = torch.concat([logits_cont, torch.cos(logits_cont[...,2]).unsqueeze(-1)], dim = -1)
print(logits_cont_3d)


tensor([[[1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1.0000, 1.0000, 1.0000, 0.5403],
         [1

In [37]:
logits_charges = torch.rand(batch_size,30,6)
truth = torch.randint(0,6,(batch_size, 30))

print(logits_charges)
print(truth)

tensor([[[8.9761e-01, 2.5491e-02, 3.6489e-01, 3.8359e-02, 4.1589e-01,
          9.5377e-01],
         [8.4005e-01, 4.8023e-01, 7.5554e-01, 1.6927e-01, 3.5464e-02,
          4.3823e-01],
         [2.0006e-01, 1.6610e-01, 3.6585e-01, 3.3457e-01, 1.2604e-01,
          8.8230e-01],
         [4.7979e-01, 3.8399e-01, 7.9198e-01, 3.3886e-01, 5.9310e-01,
          8.9021e-01],
         [4.4126e-01, 3.4168e-02, 5.6999e-01, 7.2213e-01, 9.8492e-01,
          5.9495e-01],
         [8.3847e-01, 6.0544e-01, 9.7585e-01, 3.3388e-01, 9.4004e-01,
          9.9251e-01],
         [5.4694e-01, 1.1411e-01, 1.5854e-01, 7.4086e-01, 5.4851e-01,
          5.6418e-02],
         [2.3795e-02, 3.4105e-01, 1.0692e-02, 6.5822e-01, 1.8979e-01,
          9.9075e-01],
         [2.2163e-01, 4.1981e-01, 5.7117e-02, 1.8220e-01, 6.3997e-01,
          7.5596e-01],
         [9.0165e-01, 4.1330e-01, 7.9380e-01, 2.9616e-03, 5.8627e-01,
          5.1612e-02],
         [8.6417e-01, 2.0546e-01, 1.6061e-01, 8.9938e-01, 2.2077e-01,


In [41]:
import torch.nn as nn
Cross = nn.CrossEntropyLoss(ignore_index= 0)
loss = Cross(logits_charges.transpose(dim0 = -2, dim1 = -1), truth)
print(loss)

tensor(1.8420)


In [49]:
bos = torch.tensor([1,2,3])
clusters_transfo = torch.tile(bos, (batch_size,1))
print(clusters_transfo)

tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])


In [53]:
zeros = torch.zeros(batch_size,1).type(torch.bool)
zeros

tensor([[False],
        [False],
        [False],
        [False],
        [False]])

In [56]:
bool1 = torch.tensor([False,False,True,True])
bool2 = torch.tensor([True,False,True,False])
bool1 * bool2

tensor([False, False,  True, False])

In [63]:
next_charges_batch = torch.argmax(logits_charges[:,-1], dim = -1)
print(next_charges_batch.shape)
print(next_charges_batch)
#print(logits_charges)
is_done_prev = torch.zeros(batch_size).type(torch.bool)
new_eos_tokens_batch = ( (next_charges_batch == special_symbols["eos"]["CEL"]) * ~is_done_prev)
print(new_eos_tokens_batch)

torch.Size([5])
tensor([2, 4, 3, 1, 0])
tensor([False, False, False, False, False])


In [84]:
niter = 10
is_done_prev = torch.zeros(batch_size).type(torch.bool)
is_done = torch.zeros(batch_size).type(torch.bool)
tgt_key_padding =  torch.zeros(batch_size,1).type(torch.bool)
print(id(is_done) == id(is_done_prev))
logits_tot = torch.tile(torch.tensor([1]), (batch_size,1))
print(vocab_charges.get_index(special_symbols["eos"]["CEL"]))
for i in range(niter):
    logits_next_iter = torch.rand((batch_size,30,6))
    next_charges_batch = torch.argmax(logits_next_iter[:,-1], dim =-1)
    new_eos_tokens_batch = ( (next_charges_batch == vocab_charges.get_index(special_symbols["eos"]["CEL"])) * ~is_done_prev)
    print(new_eos_tokens_batch)
    if torch.count_nonzero(new_eos_tokens_batch) > 0:
        is_done[is_done_prev == False] = new_eos_tokens_batch[is_done_prev == False]
    next_charges_batch[is_done_prev] =0
    logits_tot = torch.concat([logits_tot, next_charges_batch.unsqueeze(-1)], dim = -1)
    tgt_key_padding = torch.hstack([tgt_key_padding, is_done_prev.unsqueeze(-1)])
    print("logits_tot ===================")
    print(logits_tot)
    print("==============================")
    is_done_prev = torch.clone(is_done)
    print("tgt_key_padding ==============")
    print(tgt_key_padding)
    print("==============================")
    
print(vocab_charges.indices_to_tokens(logits_tot))



False
tensor(2)
tensor([False, False,  True,  True,  True])
logits_tot ===================
tensor([[1, 3],
        [1, 3],
        [1, 2],
        [1, 2],
        [1, 2]])
tgt_key_padding ==============
tensor([[False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False]])
tensor([False, False, False, False, False])
logits_tot ===================
tensor([[1, 3, 3],
        [1, 3, 1],
        [1, 2, 0],
        [1, 2, 0],
        [1, 2, 0]])
tgt_key_padding ==============
tensor([[False, False, False],
        [False, False, False],
        [False, False,  True],
        [False, False,  True],
        [False, False,  True]])
tensor([ True,  True, False, False, False])
logits_tot ===================
tensor([[1, 3, 3, 2],
        [1, 3, 1, 2],
        [1, 2, 0, 0],
        [1, 2, 0, 0],
        [1, 2, 0, 0]])
tgt_key_padding ==============
tensor([[False, False, False, False],
        [False, False, False, False],
        [False, False,  T

AttributeError: 'Tensor' object has no attribute 'astype'

In [88]:
dict1 = {"one":1,
        "two":2,
        "three":3}
print(len(dict1.keys()))
torch.save(dict1, "dict1.pt")
dict1_from_file = torch.load("dict1.pt")
print(type(dict1_from_file))

3
<class 'dict'>


In [24]:
import torch.nn as nn
loss = nn.MSELoss()
input = torch.randn(3, 5, requires_grad=True)
target = torch.randn(3, 5)
output = loss(input, target)
output.backward(retain_graph=True)
first_grad = input.grad
print(input.grad)
input.grad.zero_
print(input.grad)
loss_vec = output + output
#loss_vec = torch.dot(torch.tensor([1.,1.]), loss_vec)
loss_vec.backward()
print(input.grad- first_grad)


tensor([[ 0.0977, -0.1512, -0.1803, -0.2301,  0.1387],
        [-0.0774,  0.1968, -0.2973, -0.2176,  0.1116],
        [-0.1250,  0.0392, -0.0475,  0.0377,  0.1280]])
tensor([[ 0.0977, -0.1512, -0.1803, -0.2301,  0.1387],
        [-0.0774,  0.1968, -0.2973, -0.2176,  0.1116],
        [-0.1250,  0.0392, -0.0475,  0.0377,  0.1280]])
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


In [19]:
import torch
def add_pad(input, pad_tokens, size):
    nhits_input = input.shape[1]
    diff_size = size - nhits_input
    if diff_size > 0:
        pad = pad_tokens.repeat(input.shape[0], diff_size,pad_tokens.shape[0])
        print(pad)
        input = torch.cat([input, pad], dim = -1)
    return input

a = torch.randint(5,(3,4,5))
print(a)
pad_tokens = torch.zeros(5)
a = add_pad(a,pad_tokens,7)
print(a)


tensor([[[0, 2, 2, 1, 0],
         [2, 0, 4, 0, 0],
         [4, 3, 2, 0, 1],
         [4, 1, 1, 2, 0]],

        [[4, 2, 0, 2, 1],
         [4, 0, 0, 1, 0],
         [4, 1, 2, 2, 3],
         [0, 1, 3, 3, 1]],

        [[2, 3, 1, 4, 2],
         [3, 4, 2, 3, 1],
         [4, 3, 0, 1, 0],
         [3, 1, 1, 1, 0]]])


RuntimeError: Sizes of tensors must match except in dimension 2. Expected size 4 but got size 3 for tensor number 1 in the list.